# End-to-End Multi SHP Test (2020-03-01 to 2020-06-30)

Reusable execution-layer harness for multiple SHPs:
- observed THL cases (per SHP)
- Finland timeline -> interventions
- simulation run
- scaled simulated vs observed comparison


## Section A. Imports

In [ ]:
import sys
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "extensions" / "scenario_api").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate extensions/scenario_api from current working directory")

extensions_dir = repo_root / "extensions"
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api import (
    get_default_shp_region_config,
    population_scale_factor,
    default_mask_mapping_profile,
    default_contact_policy_mapping_profile,
    default_testing_tracing_mapping_profile,
    default_mask_effectiveness_profiles,
    run_region_scenario_against_observed,
    compute_r_proxy_from_incidence,
    compute_smoothed_r_proxy,
    smooth_timeseries_moving_average,
    InterventionSet,
    compile_runtime_effects,
    compile_network_multipliers,
    is_openabm_available,
)


## Section B. Configure paths and dates

In [ ]:
observed_cases_path = repo_root / "data" / "processed" / "thl_cases_2020_2022_processed_daily.csv"
timeline_processed_path = repo_root / "data" / "processed" / "oxcgrt_finland_2020_2022_timeline.csv"

reference_start_date = "2020-03-01"
end_date = "2020-06-30"
start_date = reference_start_date
initial_infected = 100
generation_interval = 5
strict_runtime_updates = True

simulation_steps = (date.fromisoformat(end_date) - date.fromisoformat(reference_start_date)).days + 1

print("Observed path:", observed_cases_path)
print("Timeline path:", timeline_processed_path)
print("Date window:", reference_start_date, "->", end_date)
print("Simulation steps:", simulation_steps)
print("initial_infected:", initial_infected)
print("generation_interval:", generation_interval)
print("strict_runtime_updates:", strict_runtime_updates)


## Section C. Build mapping profiles

In [ ]:
mask_mapping_profile = default_mask_mapping_profile()
contact_mapping_profile = default_contact_policy_mapping_profile()
testing_tracing_mapping_profile = default_testing_tracing_mapping_profile()
mask_profiles = default_mask_effectiveness_profiles()

print("Mask mapping profile:", mask_mapping_profile.name)
print("Contact mapping profile:", contact_mapping_profile.name)
print("Testing/tracing mapping profile:", testing_tracing_mapping_profile.name)
print("Mask effectiveness profiles:", list(mask_profiles.keys()))

## Section D. Choose SHPs

In [ ]:
regions_to_test = [
    ("Helsinki and Uusimaa", 200000),
    # ("Pirkanmaa", 120000),
    # ("North Ostrobothnia", 120000),
]
regions_to_test


## Section E. Build region configs

In [ ]:
region_configs = [get_default_shp_region_config(name, simulated_population=sim_pop) for name, sim_pop in regions_to_test]

for cfg in region_configs:
    print(
        f"{cfg.name}: real={cfg.real_population}, simulated={cfg.simulated_population}, scale={population_scale_factor(cfg):.3f}"
    )

## Section F. Run end-to-end pipeline for each SHP

In [ ]:
results = {}
use_openabm = is_openabm_available()
print("OpenABM available:", use_openabm)

for cfg in region_configs:
    bundle = run_region_scenario_against_observed(
        region_config=cfg,
        observed_cases_path=str(observed_cases_path),
        timeline_processed_path=str(timeline_processed_path),
        reference_start_date=reference_start_date,
        start_date=start_date,
        end_date=end_date,
        simulation_steps=simulation_steps,
        mask_mapping_profile=mask_mapping_profile,
        contact_mapping_profile=contact_mapping_profile,
        testing_tracing_mapping_profile=testing_tracing_mapping_profile,
        mask_profiles=mask_profiles,
        use_openabm=use_openabm,
        initial_infected=initial_infected,
        strict_runtime_updates=strict_runtime_updates,
    )
    results[cfg.name] = bundle

    meta = bundle["metadata"]
    runtime_diag = bundle.get("runtime_diagnostics", {})
    print(f"\nRegion: {cfg.name}")
    print("  window:", meta["start_date"], "->", meta["end_date"])
    print("  timeline events used:", meta["timeline_events_used"])
    print("  interventions created:", meta["interventions_created"])
    print("  observed length:", meta["observed_points"])
    print("  simulated raw length:", meta["simulated_raw_points"])
    print("  simulated scaled length:", meta["simulated_scaled_points"])
    print("  openabm_used:", meta["openabm_used"])
    print("  initial_infected:", meta.get("initial_infected"))
    print("  seed_override_applied:", meta.get("seed_override_applied"))
    print("  openabm_n_seed_infection:", meta.get("openabm_n_seed_infection"))
    print("  runtime compiled/applied/unsupported totals:",
          runtime_diag.get("compiled_runtime_effects_total", 0),
          runtime_diag.get("applied_effects_total", 0),
          runtime_diag.get("unsupported_effects_total", 0))


## Section G. Plot comparison for each SHP

In [ ]:
for region_name, bundle in results.items():
    observed_daily_cases = bundle["observed_timeseries"]
    simulated_scaled_daily_cases = bundle["simulated_scaled_timeseries"]

    obs_dates = pd.to_datetime(observed_daily_cases.times)
    obs_values = pd.Series(observed_daily_cases.values, dtype=float)

    ref_ts = pd.Timestamp(reference_start_date)
    sim_dates = [ref_ts + pd.Timedelta(days=int(t)) for t in simulated_scaled_daily_cases.times]
    sim_values = pd.Series(simulated_scaled_daily_cases.values, dtype=float)

    n = min(len(obs_dates), len(sim_dates), len(obs_values), len(sim_values))
    obs_dates = obs_dates[:n]
    obs_values = obs_values.iloc[:n]
    sim_dates = sim_dates[:n]
    sim_values = sim_values.iloc[:n]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(obs_dates.to_numpy(), obs_values.to_numpy(), label=f"Observed THL daily ({region_name})", linewidth=2)
    ax.plot(pd.Series(sim_dates).to_numpy(), sim_values.to_numpy(), label="Simulated daily (scaled)", linewidth=2)
    ax.set_title(f"{region_name}: Daily cases (linear)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Cases/day")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


## Section H. Compact summary table

In [ ]:
rows = []
for region_name, bundle in results.items():
    cfg = bundle["region_config"]
    meta = bundle["metadata"]
    observed_peak = float(max(bundle["observed_timeseries"].values)) if bundle["observed_timeseries"].values else float("nan")
    simulated_peak = float(max(bundle["simulated_scaled_timeseries"].values)) if bundle["simulated_scaled_timeseries"].values else float("nan")
    runtime_diag = bundle.get("runtime_diagnostics", {})
    rows.append(
        {
            "region": region_name,
            "real_population": cfg.real_population,
            "simulated_population": cfg.simulated_population,
            "scale_factor": meta["scale_factor"],
            "observed_daily_peak": observed_peak,
            "simulated_scaled_daily_peak": simulated_peak,
            "runtime_compiled_total": runtime_diag.get("compiled_runtime_effects_total", 0),
            "runtime_applied_total": runtime_diag.get("applied_effects_total", 0),
            "runtime_unsupported_total": runtime_diag.get("unsupported_effects_total", 0),
            "seed_override_applied": meta.get("seed_override_applied"),
        }
    )

summary_df = pd.DataFrame(rows).sort_values("region").reset_index(drop=True)
summary_df


## Section I. Short interpretation

In [ ]:
hus_name = "Helsinki and Uusimaa Hospital District"
if hus_name not in results:
    raise RuntimeError(f"{hus_name} not found in run results")

hus_bundle = results[hus_name]
hus_meta = hus_bundle["metadata"]
hus_runtime_diag = hus_bundle.get("runtime_diagnostics", {})

print("HUS runtime summary")
print("  active interventions total:", hus_runtime_diag.get("active_interventions_total", 0))
print("  compiled runtime effects total:", hus_runtime_diag.get("compiled_runtime_effects_total", 0))
print("  applied effects total:", hus_runtime_diag.get("applied_effects_total", 0))
print("  unsupported effects total:", hus_runtime_diag.get("unsupported_effects_total", 0))
print("  sample applied effects:")
for row in hus_runtime_diag.get("applied_effect_samples", [])[:5]:
    print("   ", row)
print("  sample unsupported effects:")
for row in hus_runtime_diag.get("unsupported_effect_samples", [])[:5]:
    print("   ", row)

# R-proxy diagnostics
simulated_scaled_daily_cases = hus_bundle["simulated_scaled_timeseries"]
r_proxy = compute_r_proxy_from_incidence(simulated_scaled_daily_cases, generation_interval=generation_interval)

r_dates = [pd.Timestamp(reference_start_date) + pd.Timedelta(days=int(t)) for t in r_proxy.times]
r_values = pd.Series(r_proxy.values, dtype=float)
r_values_ma7 = r_values.rolling(window=7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pd.Series(r_dates).to_numpy(), r_values.to_numpy(), label=f"R proxy (GI={generation_interval})", linewidth=1.5, alpha=0.45)
ax.plot(pd.Series(r_dates).to_numpy(), r_values_ma7.to_numpy(), label="R proxy MA(7)", linewidth=2.5)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="R = 1")
ax.set_title("HUS: simulated R-proxy over time")
ax.set_xlabel("Date")
ax.set_ylabel("R proxy")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

# Runtime-effect sanity check at key periods
intervention_set = InterventionSet(hus_bundle["interventions"])
period_samples = [(5, "early"), (25, "lockdown-ish"), (80, "later")]
print("\nRuntime-effect sanity check (HUS)")
for day_idx, label in period_samples:
    compiled = compile_runtime_effects(intervention_set, t=day_idx)
    targets = {e.target: e.value for e in compiled.get("applied_effects", [])}
    print(f"  t={day_idx:>3} ({label}):", targets)


## R diagnostics: R0 vs R_obs vs R_agent

In [ ]:
hus_name = "Helsinki and Uusimaa Hospital District"
if hus_name not in results:
    raise ValueError(f"{hus_name} not found in results")

hus_bundle = results[hus_name]
observed_daily_cases = hus_bundle.get("observed_timeseries")
simulated_daily_cases = hus_bundle.get("simulated_raw_timeseries")

if observed_daily_cases is None or len(observed_daily_cases.values) == 0:
    raise ValueError("Missing observed daily cases for R_obs diagnostics")
if simulated_daily_cases is None or len(simulated_daily_cases.values) == 0:
    raise ValueError("Missing simulated daily incidence for R_agent diagnostics")

R0_target = 2.3
generation_interval = 5
smoothing_window = 7
network_weights = {"work": 0.35, "school": 0.15, "random": 0.50}

print("SHP:", hus_name)
print("Date window:", reference_start_date, "->", end_date)
print("R0_target:", R0_target)
print("generation_interval:", generation_interval)
print("smoothing_window:", smoothing_window)
print("R_linear weights:", network_weights)

R_obs_raw = compute_r_proxy_from_incidence(observed_daily_cases, generation_interval=generation_interval, new_name="R_obs_raw")
R_obs_smoothed = compute_smoothed_r_proxy(observed_daily_cases, generation_interval=generation_interval, smoothing_window=smoothing_window, new_name="R_obs_smoothed")

R_agent_raw = compute_r_proxy_from_incidence(simulated_daily_cases, generation_interval=generation_interval, new_name="R_agent_raw")
R_agent_smoothed = compute_smoothed_r_proxy(simulated_daily_cases, generation_interval=generation_interval, smoothing_window=smoothing_window, new_name="R_agent_smoothed")

# R_linear: lineaarinen arvio R0_target-arvosta ja verkkomultipliereista
intervention_set = InterventionSet(hus_bundle["interventions"])
n_days = len(simulated_daily_cases.values)
r_linear_values = []
for t in range(n_days):
    multipliers = compile_network_multipliers(intervention_set, t=t)
    total_weight = 0.0
    weighted_sum = 0.0
    for network, weight in network_weights.items():
        if float(weight) <= 0:
            continue
        total_weight += float(weight)
        weighted_sum += float(weight) * float(multipliers.get(network, 1.0))
    aggregate_multiplier = (weighted_sum / total_weight) if total_weight > 0 else 1.0
    r_linear_values.append(float(R0_target) * float(aggregate_multiplier))

r_linear_raw = simulated_daily_cases.__class__(
    name="R_linear_raw",
    times=list(simulated_daily_cases.times),
    values=r_linear_values,
    variable="r_linear",
    source_type="diagnostic",
    source_name="R0_target_x_network_multiplier",
    metadata={
        "method": "R0_target_times_weighted_network_multiplier",
        "R0_target": float(R0_target),
        "network_weights": dict(network_weights),
    },
)
R_linear_smoothed = smooth_timeseries_moving_average(r_linear_raw, window=smoothing_window, new_name="R_linear_smoothed")

print("\nR_obs first valid values:")
print(pd.Series(R_obs_raw.values, dtype=float).dropna().head(5).to_list())
print("R_obs metadata:", R_obs_smoothed.metadata)

print("\nR_agent first valid values:")
print(pd.Series(R_agent_raw.values, dtype=float).dropna().head(5).to_list())
print("R_agent metadata:", R_agent_smoothed.metadata)

print("\nR_linear first values:")
print(pd.Series(R_linear_smoothed.values, dtype=float).head(5).to_list())
print("R_linear metadata:", R_linear_smoothed.metadata)


In [ ]:
obs_dates = pd.to_datetime(observed_daily_cases.times)
obs_df = pd.DataFrame({
    "date": obs_dates,
    "R_obs_smoothed": pd.Series(R_obs_smoothed.values, dtype=float).to_numpy(),
})

agent_dates = pd.to_datetime([pd.Timestamp(reference_start_date) + pd.Timedelta(days=int(t)) for t in simulated_daily_cases.times])
agent_df = pd.DataFrame({
    "date": agent_dates,
    "R_agent_smoothed": pd.Series(R_agent_smoothed.values, dtype=float).to_numpy(),
})

linear_dates = pd.to_datetime([pd.Timestamp(reference_start_date) + pd.Timedelta(days=int(t)) for t in R_linear_smoothed.times])
linear_df = pd.DataFrame({
    "date": linear_dates,
    "R_linear_smoothed": pd.Series(R_linear_smoothed.values, dtype=float).to_numpy(),
})

r_compare = (
    obs_df
    .merge(agent_df, on="date", how="outer")
    .merge(linear_df, on="date", how="outer")
    .sort_values("date")
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(r_compare["date"].to_numpy(), r_compare["R_obs_smoothed"].to_numpy(), label="R_obs (smoothed, MA7)", linewidth=2)
ax.plot(r_compare["date"].to_numpy(), r_compare["R_agent_smoothed"].to_numpy(), label="R_agent (smoothed, MA7)", linewidth=2)
ax.plot(r_compare["date"].to_numpy(), r_compare["R_linear_smoothed"].to_numpy(), label="R_linear (R0 x network, MA7)", linewidth=2)
ax.axhline(R0_target, color="tab:gray", linestyle="--", linewidth=1.8, label=f"R0_target = {R0_target}")
ax.axhline(1.0, color="black", linestyle=":", linewidth=1.5, label="R = 1")
ax.set_title(f"{hus_name}: R diagnostics ({reference_start_date} to {end_date})")
ax.set_xlabel("Date")
ax.set_ylabel("R indicator (smoothed)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
valid = r_compare.dropna(subset=["R_obs_smoothed", "R_agent_smoothed", "R_linear_smoothed"]).copy()
if valid.empty:
    raise ValueError("R diagnostics comparison has no overlapping valid points")

early = valid.head(min(21, len(valid)))
later = valid.tail(min(21, len(valid)))

obs_early = float(early["R_obs_smoothed"].mean())
obs_later = float(later["R_obs_smoothed"].mean())
agent_early = float(early["R_agent_smoothed"].mean())
agent_later = float(later["R_agent_smoothed"].mean())
linear_early = float(early["R_linear_smoothed"].mean())
linear_later = float(later["R_linear_smoothed"].mean())

print("Diagnostic interpretation")
print(f"- Baseline reference R0_target = {R0_target:.2f}")
print(f"- Early means: R_obs={obs_early:.2f}, R_agent={agent_early:.2f}, R_linear={linear_early:.2f}")
print(f"- Late means:  R_obs={obs_later:.2f}, R_agent={agent_later:.2f}, R_linear={linear_later:.2f}")

if agent_early > R0_target * 1.2:
    print("- Main mismatch: baseline/tartuttavuus vaikuttaa liian korkealta (R_agent yli R0_target alkuvaiheessa).")
elif agent_early < R0_target * 0.8:
    print("- Main mismatch: baseline/tartuttavuus vaikuttaa liian matalalta (R_agent alle R0_target alkuvaiheessa).")
else:
    print("- Baseline-taso: R_agent on alkuvaiheessa kohtuullisen lähellä R0_target-arvoa.")

if agent_later < agent_early:
    print("- Interventiosignaali: R_agent laskee ajan myötä, eli rajoituksilla on suuntavaikutus.")
else:
    print("- Interventiosignaali: R_agent ei laske selvästi; vaikutus voi olla liian heikko tai ajoitus pielessä.")

if (obs_later - obs_early) * (agent_later - agent_early) >= 0:
    print("- Suuntatulkinta: R_agent seuraa R_obs-käyrän suuntaa ainakin karkeasti.")
else:
    print("- Suuntatulkinta: R_agent ja R_obs liikkuvat eri suuntiin; timing/magnitude-mismatch mahdollinen.")

if abs(agent_early - linear_early) <= 0.25 * max(1.0, linear_early):
    print("- R_linear-vertailu: R_agent on alkuvaiheessa melko linjassa lineaarisen verkkoarvion kanssa.")
else:
    print("- R_linear-vertailu: R_agent poikkeaa lineaarisesta verkkoarviosta; ei-lineaarisuus/stokastiikka tai parametrit vaikuttavat.")
